# Walmart Time Series Forecasting — Feature Engineering

## Objectives

The objective of this notebook is to clean/transform the data and engineering features motivated by `01_eda.ipynb` findings. 

## Outputs

Cleaned, transformed, and scaled dataset saved to `data/processed/`. `X_train.csv`, `X_test.csv`, `y_train.csv`, and `y_test.csv` splits defered to modeling in `03_modeling_and_evaluation.ipynb`.

## 2.1 Setup & Imports

Importing `pandas`, same as in `01_eda.ipynb`, as well as `numpy` for log transforms and array operations for lag features. Path constants defined here to ensure reproducibility and consistent file references throughout the notebook.

In [80]:
import numpy as np
import pandas as pd 

PROCESSED_DATA_PATH = '../data/processed/walmart_merged.csv'

### Load Dataset

In [81]:
df = pd.read_csv(PROCESSED_DATA_PATH)
df['Date'] = pd.to_datetime(df['Date'])

## 2.2 Data Cleaning
In this section: I handle the negative values found in `Weekly_Sales` and `MarkDown2/3`, Encode for missing values in `MarkDown1-5`, and transform the skewed features. 

### 2.2.1 Handling Negative Values
To determine whether the negative values are real values or just data entry anaomalies, I filter the dataframe to only the negative values for each of the features, then use `.describe()` on the filtered df to check if the negatives are small anomolies or normal-sized values with flipped sign. Then I use .`value_counts().head()` on `Store` and `Dept` to check if the negative values are concentrated in a few stores/departments or evenly spread.

In [14]:
neg_sales = df[df['Weekly_Sales'] < 0]
print(neg_sales['Weekly_Sales'].describe())
print(neg_sales['Store'].value_counts().head())
print(neg_sales['Dept'].value_counts().head())


count    1285.000000
mean      -68.608218
std       231.664245
min     -4988.940000
25%       -41.000000
50%       -13.200000
75%        -4.940000
max        -0.020000
Name: Weekly_Sales, dtype: float64
Store
35    124
18     52
10     50
17     49
15     45
Name: count, dtype: int64
Dept
47    254
18    180
54    146
19     87
94     77
Name: count, dtype: int64


The negative `Weekly_Sales` values are spread across many different `Store`s/`Dept`s. Since there is no dominant cluster, this is likely not an error, but a real signal consistent with returns exceeding sales. 

In [15]:
neg_md2 = df[df['MarkDown2'] < 0]
print(neg_md2['MarkDown2'].describe())
print(neg_md2['Store'].value_counts().head())
print(neg_md2['Dept'].value_counts().head())

count    1311.000000
mean      -30.862517
std        70.446056
min      -265.760000
25%       -10.500000
50%        -7.010000
75%        -2.000000
max        -0.600000
Name: MarkDown2, dtype: float64
Store
41    145
10    142
15    138
9     124
20     72
Name: count, dtype: int64
Dept
1    19
2    19
3    19
4    19
5    19
Name: count, dtype: int64


The negative values for `MarkDown2`, unlike `Weekly_Sales`, has `Dept` `.value_counts()` identical across every department (19) which is highly suspicious since the negative values don't appear to be an independent per-`Dept` event. I will check whether `MarkDown2` is actually measured per-`Dept` or by some other measure. 

To determine if `MarkDown2` is measured per-`Dept` or some other measure, I will take the first `notna()` `MarkDown2` `Store` and `Date`, and check the store-week's mark down value. 

In [8]:
sample = df[df['MarkDown2'].notna()][['Store', 'Date']].iloc[0]
df[(df['Store'] == sample['Store']) & (df['Date'] == sample['Date'])][['Dept', 'MarkDown2']]

,Dept,MarkDown2
92,1,6115.67
235,2,6115.67
378,3,6115.67
521,4,6115.67
664,5,6115.67
...,...,...
9605,94,6115.67
9748,95,6115.67
9870,96,6115.67
10013,97,6115.67


The same `MarkDown2` value (6115.67) is repeated across all `Dept`s for the given store-week. This means that mark down values are recorded at the store-week level and then broadcast onto every `Dept` row, not per department. 

Since the mark down values are recorded at the store-week level, I will check the negative `MarkDown2` values with dropped duplicated.

In [5]:
df[df['MarkDown2'] < 0][['Store', 'Date']].drop_duplicates()

,Store,Date
29629,4,2012-03-23
39922,5,2012-08-17
78788,9,2012-08-10
78792,9,2012-09-07
87634,10,2012-03-16
87652,10,2012-07-20
108027,12,2012-07-06
128207,14,2012-07-13
138253,15,2012-08-24
138262,15,2012-10-26


This shows there are actually only 19 distinct store-week events with negitive mark downs, not 1311 independent rows found earlier. This means the `Dept`-level negative counts were inflated by the broadcasting when dataframes were merged. 

To determine if the negative `MarkDown2` values are real signals or data entry errors, I will check the values nearby, in time, of a store that had a negative `MarkDown2` to see whether those nearby `MarkDown2` values look disconnected from the negative value or if the negative value's magnitude roughly relates to nearby positive value, which would suggest a correction/clawback rather than a data entry error. 

In [ ]:
sample_store = (df[df['Store'] == 4].drop_duplicates(subset=['Date'])).sort_values(by='Date').reset_index()

i_neg = sample_store[sample_store['Date']=='2012-03-23'].index[0]
# look at values nearby
sample_store[['Date','MarkDown2']].iloc[i_neg-5:i_neg+5,]

,Date,MarkDown2
106,2012-02-17,11049.65
107,2012-02-24,4703.88
108,2012-03-02,1394.86
109,2012-03-09,602.02
110,2012-03-16,37.12
111,2012-03-23,-10.50
112,2012-03-30,442.52
113,2012-04-06,NaN
114,2012-04-13,5941.43
115,2012-04-20,4279.41


Since `MarkDown3` likely has the same broadcasting structure, I will skip the full `Store`-`Date` lookup, but will verify with `.describe()` and `.value_counts()` first. 

In [16]:
neg_md3 = df[df['MarkDown3'] < 0]
print(neg_md3['MarkDown3'].describe())
print(neg_md3['Store'].value_counts().head())
print(neg_md3['Dept'].value_counts().head())

count    257.000000
mean      -8.634319
std       12.796205
min      -29.100000
25%      -29.100000
50%       -1.000000
75%       -0.200000
max       -0.200000
Name: MarkDown3, dtype: float64
Store
28    72
31    70
39    69
36    46
Name: count, dtype: int64
Dept
1    4
2    4
3    4
4    4
5    4
Name: count, dtype: int64


Again, `Dept` `.value_counts()` is identical across every department (4), confirming the same broadcast structure as `MarkDown2`. I will not run a separate nearby-values check, since I am extending the `MarkDown2` findings to `MarkDown3` by structural analogy. 

Given the negative values for `Weekly_Sales` and `MarkDown2/3` are real signals, I can't simply remove them since that would result in losing real data, but I still need to transform them since negative values can't be log transformed. Therefore, to deal with the negative values I will multiply the `.sign()` of the values with `.log1p()` of the absolute value, preserving direction while making magnitude log-scale-safe, for `Weekly_Sales` and `MarkDown2/3`.

### 2.2.2 Encoding Missing Values

From the dataset description, the missing values in the mark down columns are not unknown values, but rather instances where no promotion occured. Therefore, I encode `NaN` values with 0 and create a separate boolean flag column, so that the downstream model can distinguish a true zero (no promotion) from a real value near 0 (small promotion). 

In [82]:
markdowns = ['MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5']

for m in markdowns:
    # create flag column
    df[f'{m}_No_Promotion'] = df[m].isna().astype(int)
    # encode missing vals with 0
    df[m] = df[m].fillna(0)

print(df.columns.tolist())
# flag count should match how many NaNs existed originally
print(df['MarkDown1_No_Promotion'].sum()) 
print((df['MarkDown1'] == 0).sum())  # should be >= flagged count


['Store', 'Dept', 'Date', 'Weekly_Sales', 'IsHoliday', 'Type', 'Size', 'Temperature', 'Fuel_Price', 'MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5', 'CPI', 'Unemployment', 'MarkDown1_No_Promotion', 'MarkDown2_No_Promotion', 'MarkDown3_No_Promotion', 'MarkDown4_No_Promotion', 'MarkDown5_No_Promotion']
270889
270889


Used `df.columns.tolist()` and compared `MarkDown1_No_Promotion.sum()` and `(df['MarkDown1'] == 0).sum()` which both returned 270889, matching the number of `NaN`s found during EDA, to confirm that the five new flag markdowns were added correctly and the `NaN` values were correctly encoded.

### 2.2.3 Transform Skewed Features

To transform the skewed features (numerically determined which features to transform by absolute value of `.skew() > 0.5`) for down stream linear modeling, I use the expression `np.sign(x) * np.log1p(abs(x))` to deal with negative values, while preserving magnitude. 

In [83]:
continuous_features = ['Weekly_Sales', 'Size', 'Temperature', 'Fuel_Price', 'MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5', 'CPI', 'Unemployment']
df[continuous_features].skew()

Weekly_Sales     3.262008
Size            -0.325850
Temperature     -0.321404
Fuel_Price      -0.104901
MarkDown1        4.731304
MarkDown2       10.645956
MarkDown3       14.922341
MarkDown4        8.077666
MarkDown5        9.964519
CPI              0.085219
Unemployment     1.183743
dtype: float64

Only `Weekly_Sales`, `MarkDown1-5`, and `Unemployment` have values above the 0.5 threshold, so those will be the only features I transform.

In [84]:
skewed_features = ['Weekly_Sales', 'MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5', 'Unemployment']
for s in skewed_features:
   df[f'{s}_log'] = np.sign(df[s]) * np.log1p(abs(df[s]))

In [85]:
trans_features = ['Weekly_Sales_log', 'MarkDown1_log', 'MarkDown2_log', 'MarkDown3_log', 'MarkDown4_log', 'MarkDown5_log', 'Unemployment_log']
df[trans_features].skew()

Weekly_Sales_log   -1.500279
MarkDown1_log       0.725749
MarkDown2_log       1.736505
MarkDown3_log       2.003662
MarkDown4_log       0.986896
MarkDown5_log       0.640103
Unemployment_log    0.201250
dtype: float64

I save the transformed values into new `_log` columns to preserve the original values for debugging. Either the original or `_log` columns will be dropped during modeling depending on model type. I check transformation by running `.skew()`, which shows that the log transfom reduced skew substantially but did not fully eliminate it for `MarkDown1-5`, and `Weekly_Sales_log` flipped sign — likely due to the small number of large-magnitude negative values combined with a large number of small positive values. Since this transformation is primarily relevant to a baseline linear regression (a random forest is expected to be the stronger model and doesn't require it), I will accept this limitation and move on and will note it in limitations/future-improvemnts section. 

### 2.2.4 Categorical Encoding

`Type` (`A`/`B`/`C`) is nominal with no inherent order, so one-hot encoding is used rather than label/ordinal encoding, which would falsely imply an ordering between categories.

In [86]:
df = pd.get_dummies(df, columns=['Type'], drop_first=True)

## 2.3 Feature Engineering

In this section I engineer calendar and lag features, to help the downstream model learn patterns that might otherwise be missed if left to interpratation. 

### 2.3.1 Calendar Features

I will engineer two calendar features: `days_until_holiday` and `days_since_holiday`, to capture the real world pattern of shopping events in relation to a holiday's specific dates that `IsHoliday` does not capture with its binary values. They will represent the days until/since the next/previous holiday. To do this I will use cyclical encoding on months and weeks to encode the dates circularly, so that December(12)/January(1) and (30th/31st)/1st are correctly positioned close to each other in time, which the current linear structure would not represent.

I chose to engineer two separate until/since holiday columns over one combined distance feature (negative values are days until, positive values are days since), since a combined feature would cause big jump for the moment right after `Christmas`, going from the day before, 1 day until `Christmas` to the next day being 364 until `Christmas`, which is a discontinuity and would not be a learnable pattern.

In [87]:
# Extract week/month
df['Month'] = df['Date'].dt.month
df['Week'] = df['Date'].dt.isocalendar().week.astype(int)

# Encode Month cyclically
df['Month_sin'] = np.sin(2 * np.pi * df['Month'] / 12)
df['Month_cos'] = np.cos(2 * np.pi * df['Month'] / 12)

# Encode Week cyclically 
df['Week_sin'] = np.sin(2 * np.pi * df['Week'] / 52)
df['Week_cos'] = np.cos(2 * np.pi * df['Week'] / 52)

# holiday days

holiday_days = {'Super Bowl': ['2010-02-12', '2011-02-11', '2012-02-10', '2013-02-08'],
                'Labor Day': ['2010-09-10', '2011-09-09', '2012-09-07', '2013-09-06'],
                'Thanksgiving': ['2010-11-26', '2011-11-25', '2012-11-23', '2013-11-29'],
                'Christmas': ['2010-12-31', '2011-12-30', '2012-12-28', '2013-12-29']}
# holiday dates
all_holidays = pd.to_datetime([date for dates in holiday_days.values() for date in dates]).sort_values().to_numpy()

# Index of current Date in sorted holidays list
date_idx = np.searchsorted(all_holidays, df['Date'].to_numpy())

# edge case handling (since the min and max Date datarange for this dataset falls entierly inside holiday_days clip never needs to reuse a boundary value, but this would be fragile if the dataset were extended)
next_idx = np.clip(date_idx, 0, len(all_holidays)-1)
prev_idx = np.clip(date_idx - 1, 0, len(all_holidays)-1)

next_holiday_date = all_holidays[next_idx]
prev_holiday_date = all_holidays[prev_idx]

# engineered features
df['days_until_holiday'] = (next_holiday_date - df['Date'].to_numpy()).astype('timedelta64[D]').astype(int)
df['days_since_holiday'] = (df['Date'].to_numpy() - prev_holiday_date).astype('timedelta64[D]').astype(int)

In [88]:
# Check exact match: Super Bowl 2010-02-12
check = df[df['Date'] == '2010-02-12']
print(check[['Date', 'days_until_holiday', 'days_since_holiday']].head())

# Check rows flagged as IsHoliday == True. Confirm days_until/since are consistent
holiday_rows = df[df['IsHoliday'] == True]
print(holiday_rows[['Date', 'IsHoliday', 'days_until_holiday', 'days_since_holiday']].drop_duplicates('Date'))

          Date  days_until_holiday  days_since_holiday
1   2010-02-12                   0                   0
144 2010-02-12                   0                   0
287 2010-02-12                   0                   0
430 2010-02-12                   0                   0
573 2010-02-12                   0                   0
          Date  IsHoliday  days_until_holiday  days_since_holiday
1   2010-02-12       True                   0                   0
31  2010-09-10       True                   0                 210
42  2010-11-26       True                   0                  77
47  2010-12-31       True                   0                  35
53  2011-02-11       True                   0                  42
83  2011-09-09       True                   0                 210
94  2011-11-25       True                   0                  77
99  2011-12-30       True                   0                  35
105 2012-02-10       True                   0                  42
135 2012-0

I check to make sure the new features were created correctly by checking the features at the date of the `Super Bowl` (confirmed by the 0s), and the near-holiday behavior by checking `days_until/since_holiday`, which is confirmed to be working correctly as the `days_until_holiday` are all 0, and `days_since_holiday` are consistent — `210` days between the `Super Bowl` and `Labor Day` — (with the exception of `2010-02-12` since it is a `.clip()` artifact, it's the very first holiday in the dataset date range. This is accepted since it affects only rows for the single earliest date, which is a negligible fraction of the dataset, and avoids intorducing `NaN`s or a distorting sentinel value). 

### 2.3.2 Lag Features

Lag features provide the model with past values of the target itself as an input feature allowing the model to account for changes in sales due to momentum or seasonality. 

I will be engineering two lag features, a one week, short term lag feature to capture any sales increases/decreases due to momentum, and a one year lag feature to capture long-term recurring patterns like seasonality. 

In [89]:
df['lag_1'] = df.groupby(['Store', 'Dept'])['Weekly_Sales'].shift(1)
df['lag_52'] = df.groupby(['Store', 'Dept'])['Weekly_Sales'].shift(52).dropna()

In [90]:
df['lag_52'].isna().sum(), df['lag_1'].isna().sum()

(np.int64(160487), np.int64(3331))

In [91]:
df = df.dropna(subset=['lag_1', 'lag_52'])
print(df.shape)
df['lag_52'].isna().sum(), df['lag_1'].isna().sum()

(261083, 39)


(np.int64(0), np.int64(0))

After creating the two lag features, there were expectedly a number of `NaN` rows, since the beginning `Date`s would have no prior year or week. I chose to drop the `NaN` for both `lag_52` and `lag_1` even though `lag_52` had a significant number of missing values (160,487/421,570) because fabricating a year-ago value for data that doesnt exist isn't defensible. Additionally I dropped these before defining the temporal split to avoid inconsistent treatment between train/test. 

## 2.4 Temporal Train/Test Split

Since the data is temporal, I can't use a random `train_test_split()` since doing so would result in the model training on future data that occur chronologically after the ones it's tested on, which is a leakage the model would never encounter in real deployment. 

To do this split I will split on a globally fixed calendar date (`2011-12-31`) since the data is grouped by `Store`/`Dept` it avoids the assumption of the two different categories having roughly the same number of rows across the same date range, that splitting by a global percentage would do. 

By setting the cutoff at `2011-12-31` the train/test proportions become (52%/48%), because of the dataset date range, the dataset only contains one full Nov/Dec cycle, meaning any split either gives train or test the holiday data but not both. I prioritize giving the model that signal in training rather than being able to evaluate holiday performance directly in testing. This is a major limitation that I will document in modeling and README. 

In [92]:
train = df[df['Date'] <= '2011-12-31']
test = df[df['Date'] > '2011-12-31']

In [93]:
# sanity check
train.shape, test.shape

((136342, 39), (124741, 39))

## 2.5 Context Variable Decision

I will defer dropping context variables (`CPI`, `Unemployment`, `Fuel_Price`, `Type`, `Size`) to feature importance in modeling, since linear correlation can miss nonlinear relationships a random forest could pick up.

## 2.6 Export

I save the train and test to `../data/processed` to be loaded in modeling, but I will do the X/y split in modeling since I need to drop either `Weekly_Sales_log` or `Weekly_Sales` for random forest and linear regression respectively.  

In [94]:
train.to_csv('../data/processed/train.csv', index=False)
test.to_csv('../data/processed/test.csv', index=False)

## 2.7 Decisions Table

## Decisions Table — 02_feature_engineering.ipynb

| Section | Decision | Reasoning |
|---|---|---|
| Data Cleaning | Negative values in `Weekly_Sales` and `MarkDown1-5` columns treated as real, structural values (not data anomalies) | Negative sales reflect returns; negative markdowns reflect store-level correction/clawback events — not data entry errors |
| Data Cleaning | Applied `np.sign(x) * np.log1p(abs(x))` to skewed columns (`MarkDown1-5`, `Weekly_Sales`, `Unemployment`) | Preserves sign/magnitude while allowing log transform to work on values that include negatives, which standard `log1p` can't handle |
| Data Cleaning | Encoded `MarkDown1-5` `NaN`s as 0, and added boolean flag column | A missing markdown (no promotion) and a near-zero markdown are semantically different — flag preserves that distinction instead of conflating them |
| Categorical Encoding | One-hot encoded `Type` (`A`/`B`/`C`), `drop_first=True` | `Type` is nominal with no inherent order, so one-hot avoids falsely implying ordering that label/ordinal encoding would introduce. Done before the temporal split so train/test share identical dummy columns |
| Feature Engineering | Encoded `Month` and `Week` as sin/cos pairs (`Month_sin/cos`, `Week_sin/cos`) using `period=12` and `period=52` | Raw month/week numbers treat December and January as maximally distant; cyclical encoding correctly represents them as adjacent |
| Feature Engineering | Engineered `days_until_holiday` / `days_since_holiday` using `np.searchsorted()` on sorted holiday dates, with `np.clip()` on boundary indices | Captures shopping ramp-up/wind-down around specific holiday dates that the binary `IsHoliday` feature misses; two directional features avoid a false discontinuity a single signed-distance feature would create at the holiday boundary |
| Feature Engineering | Accepted a known `.clip()` boundary artifact: earliest holiday in the dataset gets `days_since_holiday=0` despite no true prior holiday existing in range | Affects exactly one date; `NaN`/sentinel alternatives were worse for `RandomForest` input than a documented, bounded artifact |
| Feature Engineering | Engineered `lag_1` (prior week) and `lag_52` (year-ago same week) using `groupby(['Store','Dept'])['Weekly_Sales'].shift(n)` | Captures short-term momentum and yearly seasonality; grouping before shifting is required to prevent lag values leaking across `Store`/`Dept` boundaries |
| Feature Engineering | Dropped rows with `NaN` `lag_1`/`lag_52` (3,331 and 160,487 rows respectively) rather than filling, and dropped before defining the temporal split | Fabricating a "year-ago" value where none exists isn't defensible; filling with mean/sentinel would inject fake signal. Dropping before the split (not after) avoids inconsistent train/test treatment |
| Context Variables | Deferred drop decision on `CPI`, `Unemployment`, `Fuel_Price`, `Type`, `Size` to modeling's feature-importance stage | EDA-stage linear correlation can miss nonlinear relationships that a random forest could still exploit |
| Temporal Split | Fixed global calendar cutoff at `2011-12-31` (52%/48% train/test) | Random split would leak future into train; percentage-based split assumes uniform row counts per `Store`/`Dept`, which doesn't hold. Chosen cutoff prioritizes giving train the only full Nov/Dec holiday cycle in the dataset, since test evaluating holiday performance directly wasn't possible either way — documented as a limitation |
| Export | X/y split deferred to modeling notebook; will drop `Weekly_Sales` for linear regression (uses `Weekly_Sales_log`) and drop `Weekly_Sales_log` for random forest (uses `Weekly_Sales`) | Target/feature set differs by model, so splitting now would be premature |